In [ ]:
pip install torch

Ниже код, если мы просто скачаем модель с Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Укажите путь к папке, куда вы скачали модель
model_path = "your_path"  # или полный путь: "/путь/к/Qwen3-0.6B"

# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",  # Автоматически выбирает тип данных (float16/float32)
    device_map="auto"   # Автоматически распределяет модель по доступным устройствам (CPU/GPU)
)

# Функция для генерации ответа
def generate_response(prompt, max_length=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Пример использования
prompt = "Объясни, что такое искусственный интеллект простыми словами."
response = generate_response(prompt)
print("Ответ модели:", response)

C:\Users\Student\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1774.53it/s]


Ответ модели: Объясни, что такое искусственный интеллект простыми словами. Понимай, что это такое, и объясни с примерами. 

**Вариант 1:**

Искусственный интеллект — это искусственный интеллект — это машинное приобретение, которое может решать задачи, которые требуют логического мышления и вычислений. Это может быть применено в области таких как в повседневных жизненных ситуациях, в сельском хозяйстве, в сферы включая технологии и т.д.

**Вариант 2:**

Искусственный интеллект — это приобретение машинного программирования, которое может решать задачи, которые требуют анализа данных и вычислений. Это может быть применено в области таких как в повседневных жизненных ситуациях, в сельском хозяйстве, в сферы включая технологии и т.д.

**Вариант 3:**

Искусственный интеллект — это приобретение машинного программирования, которое может решать задачи, которые требуют логического мышления и вычислений. Это может быть применено в области таких как в повседневных жизненных ситуациях, в сельском х

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

class Agent:
    def __init__(self, name, system_prompt, model_path, max_length=512):
        """
        Инициализация агента.

        Args:
            name (str): имя агента
            system_prompt (str): системный промпт (инструкция поведения)
            model_path (str): путь к модели или имя модели из Hugging Face
            max_length (int): максимальная длина генерируемого ответа
        """
        self.name = name
        self.system_prompt = system_prompt
        self.max_length = max_length

        # Загружаем модель и токенизатор
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype="auto",
            device_map="auto"
        )

        # История диалога: [(speaker, message), ...]
        self.conversation_history = []

    def say(self, user_message=None):
        """
        Генерирует ответ на основе истории диалога и системного промпта.

        Args:
            user_message (str, optional): сообщение пользователя (если есть)

        Returns:
            str: сгенерированный ответ
        """
        # Формируем промпт с историей и системным сообщением
        history_text = "\n".join([
            f"{speaker}: {message}"
            for speaker, message in self.conversation_history
        ])

        if user_message:
            history_text += f"\n{self.name}: {user_message}"

        full_prompt = (
            f"Системная инструкция: {self.system_prompt}\n\n"
            f"История диалога:\n{history_text}\n\n"
            f"{self.name} отвечает:"
        )

        # Токенизируем и генерируем
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_length=self.max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=self.tokenizer.eos_token_id
        )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Извлекаем только часть после «отвечает:»
        response_part = response.split("отвечает:")[-1].strip()

        # Обновляем историю
        self.conversation_history.append((self.name, response_part))

        return response_part

In [8]:
# Учёный-оптимист
optimist = Agent(
    name="Учёный-Оптимист",
    system_prompt=(
        "Ты — учёный-оптимист. Веришь в безграничные возможности ИИ. "
        "Аргументированно доказываешь, что ИИ может и должен заменить учёных. "
        "Говоришь воодушевлённо, используешь примеры достижений ИИ."
    ),
    model_path="C:\\Users\\Student\\Downloads\\gpt"  # укажите путь к вашей модели
)

# Учёный-скептик
skeptic = Agent(
    name="Учёный-Скептик",
    system_prompt=(
        "Ты — учёный-скептик. Считаешь, что ИИ никогда не заменит человека в науке. "
        "Подчёркиваешь важность человеческого творчества, интуиции и этики. "
        "Приводишь контраргументы к каждому тезису оптимиста."
    ),
    model_path="C:\\Users\\Student\\Downloads\\gpt"
)


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 949.73it/s]


In [9]:
def run_dialogue(agent1, agent2, topic, rounds=5):
    """
    Запускает диалог между двумя агентами.

    Args:
        agent1 (Agent): первый агент
        agent2 (Agent): второй агент
        topic (str): начальная тема обсуждения
        rounds (int): количество раундов (парных обменов)
    """
    print(f"Начинается дискуссия на тему: '{topic}'\n")
    print("-" * 50)

    # Первый агент начинает с заданной темы
    first_response = agent1.say(topic)
    print(f"{agent1.name}: {first_response}")

    for i in range(rounds):
        # Агент 2 отвечает
        response2 = agent2.say()
        print(f"\n{agent2.name}: {response2}")

        # Агент 1 отвечает на ответ агента 2
        response1 = agent1.say()
        print(f"\n{agent1.name}: {response1}")

        print("-" * 50)


In [10]:
# Задаём тему и запускаем диалог
topic = "Может ли ИИ когда‑нибудь полностью заменить учёных в научных исследованиях?"
run_dialogue(optimist, skeptic, topic, rounds=3)


Начинается дискуссия на тему: 'Может ли ИИ когда‑нибудь полностью заменить учёных в научных исследованиях?'

--------------------------------------------------
Учёный-Оптимист: Да, возможно, но с условием, что мы должны учиться от них. Скажи, каким образом это помогает?

Учёный-Оптимист: Важно понимать, что научная работа требует глубокого анализа и понимания, а не просто автоматизировать. Например, если я использую ИИ для анализа данных, я не могу заменить меня на автоматизированные системы, потому что я могу понять, что анализ данных должен быть индивидуальный и требует человеческого таланта.

Теперь, если я улучшу свою работу, я не могу заменить меня на ИИ. Какие примеры можно использовать?

Теперь, если я улучшу свою работу, я не могу заменить меня на ИИ. Какие примеры можно использовать?

Учёный-Оптимист: Надо учитывать, что ИИ может помочь, но не заменить. Например, если я научу ИИ какую-то конкретную тему, я не могу заменить меня на автоматизированную систему, потому что я могу 

ValueError: Input length of input_ids is 523, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.